In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.optim as optim
import os

from collections import namedtuple
from itertools import count
from multiprocessing import Pool

from torch.distributions import Categorical
from src.env.Wordle import WordleEnv
from src.models.ActorCritic import Actor, Critic
from src.models.Adversary import Adversary

Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))


# Initialize the environment (input subset size if necessary)
env = WordleEnv(mode="easy")

plt.ion()

# if GPU is to be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

torch.set_default_device(device)

MPS device not found.


In [4]:
import torch
help(torch.load)

Help on function load in module torch.serialization:

load(f: Union[str, os.PathLike[str], IO[bytes]], map_location: Union[Callable[[torch.types.Storage, str], torch.types.Storage], torch.device, str, dict[str, str], NoneType] = None, pickle_module: Any = None, *, weights_only: Optional[bool] = None, mmap: Optional[bool] = None, **pickle_load_args: Any) -> Any
    load(f, map_location=None, pickle_module=pickle, *, weights_only=True, mmap=None, **pickle_load_args)

    Loads an object saved with :func:`torch.save` from a file.

    :func:`torch.load` uses Python's unpickling facilities but treats storages,
    which underlie tensors, specially. They are first deserialized on the
    CPU and are then moved to the device they were saved from. If this fails
    (e.g. because the run time system doesn't have certain devices), an exception
    is raised. However, storages can be dynamically remapped to an alternative
    set of devices using the :attr:`map_location` argument.

    If :attr:

In [9]:
list(range(1000000, 0, -100000))

[1000000,
 900000,
 800000,
 700000,
 600000,
 500000,
 400000,
 300000,
 200000,
 100000]

## Actor Critic

In [3]:
# Actor and critic instances for the protagonist
actor_p = Actor(env.state_size, env.action_size).to(device)
critic_p = Critic(env.state_size).to(device)

# Actor and critic for the antagonist
actor_a = Actor(env.state_size, env.action_size).to(device)
critic_a = Critic(env.state_size).to(device)

# Teacher agent
adversary = Adversary(vocab_path="src/data/answers.txt", alpha=0.01, epsilon=0.2)

def select_action_actor(actor, state, available_actions, action_size):
    # Create a mask tensor for previously chosen actions
    mask = torch.full((action_size,), -1e9, device=device)
    for idx in available_actions:
        mask[idx] = 0

    # Add the mask to the actor output and sample from the distribution
    actor_output = actor(state)
    masked_output = actor_output + mask
    masked_distribution = Categorical(logits=masked_output)
    return masked_distribution.sample().view(1, 1)



In [4]:
def run_episode(actor, critic, optimizer_actor, optimizer_critic, env):
    """
    Runs one episode for the protagonist/antagonist
    """
    total_reward = 0
    done = False
    while not done:
        state = torch.tensor(env.current_state, device=device)
        actor_output, value = actor(state), critic(state)
        mask = torch.full((env.action_size,), -1e9, device=device)
        for idx in env.available_actions:
            mask[idx] = 0.0

        masked_output = actor_output + mask
        dist = Categorical(logits=masked_output)
        action = dist.sample().view(1, 1)
        next_state, reward, done, won, attempts = env.step(action.item())
        next_value = critic(torch.tensor(next_state, device=device))
        td_error = reward + 0.99 * next_value * (1 - done) - value.detach()
        log_prob = dist.log_prob(action).unsqueeze(1)

        actor_loss = -log_prob * td_error.detach()
        print(actor_loss)
        optimizer_actor.zero_grad()
        actor_loss.backward()
        optimizer_actor.step()

        critic_loss = td_error.pow(2)
        optimizer_critic.zero_grad()
        critic_loss.backward()
        optimizer_critic.step()

        total_reward += reward
        if done:
            break
    return total_reward

In [5]:
optimizer_actor_p = optim.Adam(actor_p.parameters(), lr=0.01)
optimizer_critic_p = optim.Adam(critic_p.parameters(), lr=0.01)

# optimizer_actor_a = optim.Adam(actor_a.parameters(), lr=0.01)
# optimizer_critic_a = optim.Adam(critic_a.parameters(), lr=0.01)



In [6]:
num_episodes = 300000
# torch.set_num_threads(1)
# torch.set_num_interop_threads(1)
optimizer_actor_p = optim.Adam(actor_p.parameters(), lr=0.01)
optimizer_critic_p = optim.Adam(critic_p.parameters(), lr=0.01)

# optimizer_actor_a = optim.Adam(actor_a.parameters(), lr=0.01)
# optimizer_critic_a = optim.Adam(critic_a.parameters(), lr=0.01)

seed = 439
np.random.seed(seed)
torch.manual_seed(seed)
for episode in range(num_episodes):
    # adversary_action = adversary.select_action()
    # target = env.words[adversary_action]

    env.reset()
    state = env.get_state()
    # env.target_word = target
    print(f"Episode: {episode}/{num_episodes}, Teacher selected target: {env.target_word}")
    reward_p = run_episode(actor_p, critic_p, optimizer_actor_p, optimizer_critic_p, env=env)

    # env.reset()
    # state = env.get_state()
    # env.target_word = target
    # reward_a = run_episode(actor_a, critic_a, optimizer_actor_a, optimizer_critic_a, env=env)
    #
    # regret = reward_a - reward_p # Regret is defined as the difference between the two agents' total rewards
    # adversary.observe(action=adversary_action, reward=regret)



Episode: 0/300000, Teacher selected target: LEAFY
tensor([[[18.8259]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[0.5191]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[19.9270]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[1.7962]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[39.1358]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[-433.2869]]], device='cuda:0', grad_fn=<MulBackward0>)
Episode: 1/300000, Teacher selected target: WHOLE
tensor([[[9.7008]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[10.9749]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[13.0210]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[43.8965]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[21.6431]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[-408.4887]]], device='cuda:0', grad_fn=<MulBackward0>)
Episode: 2/300000, Teacher selected target: INTRO
tensor([[[19.0194]]], device='cuda:0', grad_fn=<MulBackward0>)
tensor([[[14.0001]]], device='

KeyboardInterrupt: 

In [12]:
import json
pwd
reward_history = json.load(open('reward_history_100.json'))

NameError: name 'pwd' is not defined

In [ ]:
total_attempts = 0
correct_guesses = 0

no_test_trials = 30000

eps_threshold = 1e-11 #For only exploration

for episode in range(no_test_trials):
    state = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    for t in count():
        action = select_action_actor(actor_p, state, env.available_actions, env.action_size)
        observation, reward, done, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        
        if done:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        state = next_state
        total_attempts += 1

        if done:
            if reward[0] == 10:
                correct_guesses += 1
            break

success_rate = correct_guesses / no_test_trials
average_attempts = total_attempts / no_test_trials

print(f"Trials: {no_test_trials}, Success rate: {success_rate:.2f}, Average number of attempts: {average_attempts:.2f}")

In [ ]:
total_attempts = 0
correct_guesses = 0

no_test_trials = 30000

eps_threshold = 1e-11 #For only exploration

for episode in range(no_test_trials):
    state = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    
    action = torch.tensor([[345]], device=device, dtype=torch.long) #Salet start.
    for t in count():
        observation, reward, done, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        
        if done:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        state = next_state
        total_attempts += 1

        if done:
            if reward[0] == 10:
                correct_guesses += 1
            break
        action = select_action_actor(actor_p, state, env.available_actions, env.action_size)

success_rate = correct_guesses / no_test_trials
average_attempts = total_attempts / no_test_trials

print(f"Trials with SALET start: {no_test_trials}, Success rate: {success_rate:.2f}, Average number of attempts: {average_attempts:.2f}")

In [ ]:
state = env.reset()
env.target_word = 'OTHER'
state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
action = torch.tensor([[344]], device=device, dtype=torch.long) #Salet start.

for t in count():
    observation, reward, done, _ = env.step(action.item())
    reward = torch.tensor([reward], device=device)
    env.render()
    if done:
        next_state = None
    else:
        next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
        
    if done:
        break
    action = select_action_actor(actor_p, state, env.available_actions, env.action_size)
    state = next_state

# CL part

In [ ]:
# Same as before but added variables to plot results

def run_episode_cl(actor, critic, optimizer_actor, optimizer_critic, state, env):
    log_probs = []
    values = []
    rewards = []
    masks = []
    final_reward = 0
    episode_won = False
    
    for i in count():
        actor_output, value = actor(state), critic(state)
        mask = torch.full((env.action_size,), -1e9, device=device)
        for idx in env.available_actions:
            mask[idx] = 0.0

        masked_output = actor_output + mask
        dist = Categorical(logits=masked_output)
        action = dist.sample().view(1, 1)
        
        # next_state, reward, done, _ = env.step(action.item())

        next_state, reward, done, info = env.step(action.item())
        
        # Verifying if the game is won already
        if info.get("won", False):
            episode_won = True
        
        next_state = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)
    
        log_prob = dist.log_prob(action).unsqueeze(1)
    
        log_probs.append(log_prob)
        if value.dim() > 1:
            values.append(value.squeeze(0))
        else:
            values.append(value)
            
        rewards.append(torch.tensor([reward], dtype=torch.float, device=device))
        masks.append(torch.tensor([1-done], dtype=torch.float, device=device))
    
        state = next_state
    
        if done:
            break

    # --- Actor-Critic Update ---
    returns = []
    discounted_reward = 0

    # Calculate discounted rewards
    for r, m in zip(reversed(rewards), reversed(masks)):
        discounted_reward = r + 0.99 * discounted_reward * m
        returns.insert(0, discounted_reward)

    returns = torch.cat(returns).detach()
    if values:
        values = torch.cat(values, dim=0).unsqueeze(1)
    else:
        values = torch.zeros(1, 1, device=device, requires_grad=True)    
        
    log_probs = torch.cat(log_probs)
    advantage = returns - values
    
    optimizer_actor.zero_grad()
    optimizer_critic.zero_grad()
    
    actor_loss = -(log_probs * advantage.detach()).mean()
    critic_loss = advantage.pow(2).mean()

    actor_loss.backward()
    critic_loss.backward()
    
    optimizer_actor.step()
    optimizer_critic.step()

    # return final_reward, env.attempts
    return returns[0].item(), env.attempts, episode_won

In [ ]:
os.makedirs("Metrics", exist_ok=True)

# List of datasets in increasing order of difficulty
curriculum_datasets = [
    # "data/subset_1_easy_479.txt",
    # "data/subset_2_medium_479.txt",
    # "data/subset_3_hard_479.txt"
    "src/data/dataset_1_easy.txt",
    "src/data/dataset_2_medium.txt",
    "src/data/dataset_3_hard.txt"
]

episodes_per_phase = 100 

seeds = range(2) # change the number of seeds we finally want here
for seed in seeds:
    print(f"\n--> RUNNING SEED {seed}")
    
    # Configure deterministic seeds for this run
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    episode_durations = []
    episode_rewards = []
    win_rate_history = []

    # Instantiate environments and models for each seed
    temp_env = WordleEnv(target_dataset_path=curriculum_datasets[0])
    actor = Actor(temp_env.state_size, temp_env.action_size).to(device)
    critic = Critic(temp_env.state_size, temp_env.action_size).to(device)

    optimizer_actor = optim.Adam(actor.parameters(), lr=0.001)
    optimizer_critic = optim.Adam(critic.parameters(), lr=0.001)

    for phase_idx, dataset_path in enumerate(curriculum_datasets):
        print(f"STARTING PHASE WITH: {dataset_path}")
        
        env = WordleEnv(target_dataset_path=dataset_path)
        wins = 0
        
        for episode in range(episodes_per_phase):
            state = env.reset()

            if phase_idx == 0 and episode < 3:
                print(f"\n[DEBUG SEED] Episode {episode + 1}")
                print(f"Target Word Chosen by Env: {env.target_word}")
            
            state = torch.FloatTensor(state).to(device)

            final_reward, attempts, episode_won = run_episode_cl(actor, critic, optimizer_actor, optimizer_critic, state, env)

            if phase_idx == 0 and episode < 3:
                print(f"Agent Last Guess: {env.current_guess}")
                print(f"Total Attempts Used: {attempts} | Won: {episode_won}\n")
            
            episode_rewards.append(final_reward)
            episode_durations.append(attempts)
            
            if episode_won:
                wins += 1
            
            if (episode + 1) % 100 == 0: # use 1000 for more huge episodes runs
                current_win_rate = wins / 100
                win_rate_history.append(current_win_rate)
                print(f"Seed: {seed}, Episode: {episode + 1}/{episodes_per_phase}, Attempts: {attempts}, Reward: {final_reward:.1f}, Win Rate: {current_win_rate:.2%}")
                wins = 0 

    # Save data locally to be sure that nothing is erased between each seed loop
    filename = f"Metrics/raw_data_seed_{seed}.npz"
    np.savez(
        filename,
        durations=np.array(episode_durations),
        rewards=np.array(episode_rewards),
        win_rates=np.array(win_rate_history)
    )
    print(f"Successfully saved data for Seed {seed} to {filename}")

print("\nAll Seeds Training Complete! Data safely saved on disk.")

In [ ]:
os.makedirs("Charts", exist_ok=True)

def plot_res(durations, rewards, win_rates, filename):
    plt.figure(figsize=(15, 5))
    
    # Chart 1: Win rate
    plt.subplot(1, 3, 1)
    plt.plot(range(1000, (len(win_rates)+1)*1000, 1000), win_rates, color='green', marker='o')
    plt.title('Win rate (per blocks of 1000)')
    plt.xlabel('Global Episode')
    plt.ylabel('Win Rate')
    plt.grid(True)

    # Chart 2: Sliding average of attempts
    plt.subplot(1, 3, 2)
    if len(durations) > 100:
        means = [np.mean(durations[i:i+100]) for i in range(len(durations)-100)]
        plt.plot(means)
    else:
        plt.plot(durations)
    plt.title("Average attempts (Smoothed over 100 ep)")
    plt.xlabel('Global Episode')
    plt.ylabel("Number of attempts")

    # Chart 3: Reward evolution
    plt.subplot(1, 3, 3)
    if len(rewards) > 100:
        reward_means = [np.mean(rewards[i:i+100]) for i in range(len(rewards)-100)]
        plt.plot(reward_means, color='orange')
    else:
        plt.plot(rewards, color='orange')
    plt.title('Average reward (Smoothed)')
    plt.xlabel('Global Episode')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Chart successfully saved as: {filename}")
    plt.close() # Closes the figure to free up memory

for seed in range(10):
    data_path = f"Metrics/raw_data_seed_{seed}.npz"
    
    if os.path.exists(data_path):
        print(f"Loading data for Seed {seed}...")
        # Load the compressed data from disk
        data = np.load(data_path)
        
        episode_durations = data['durations']
        episode_rewards = data['rewards']
        win_rate_history = data['win_rates']
        
        # Generate the graph for this specific seed
        chart_filename = f"Charts/test_seed_{seed}.png"
        plot_res(episode_durations, episode_rewards, win_rate_history, filename=chart_filename)
    else:
        print(f"No data found for Seed {seed} yet (File {data_path} missing). Skipping.")

print("\nPlotting process complete!")

In [ ]:
# Evaluation test as before

def select_best_action(actor, state, available_actions, action_size):
    """ Selects action deterministically (argmax) for evaluation """
    with torch.no_grad():
        mask = torch.full((1, action_size), -1e9, device=device)
        for idx in available_actions:
            mask[0, idx] = 0.0

        actor_output = actor(state)
        masked_output = actor_output + mask
        
        return masked_output.max(1)[1].view(1, 1)

total_attempts = 0
correct_guesses = 0
no_test_trials = 1000
attempts_per_episode = []
success_history = []

# Test on the hardest dataset (last in the list)
env = WordleEnv(target_dataset_path=curriculum_datasets[-1])

for episode in range(no_test_trials):
    state = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    
    # Depends on the index associate with the word we can to test.
    # I take 345 = LOGON here
    action = torch.tensor([[345]], device=device, dtype=torch.long) 
    
    for t in count():
        # Capturing 'info' dictionary to read the "won" key
        observation, reward, done, info = env.step(action.item())
        
        if done:
            # Check if won via info dict, fallback to reward check if not present
            episode_won = info.get("won", reward == 10)
            
            if episode_won:
                correct_guesses += 1
                success_history.append('Success')
                attempts_per_episode.append(env.attempts)
            else:
                success_history.append('Failure')
                attempts_per_episode.append(env.max_attempts) # Max allowed attempts on loss
                
            total_attempts += env.attempts
            break
            
        next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
        state = next_state
        action = select_best_action(actor, state, env.available_actions, env.action_size)

success_rate = correct_guesses / no_test_trials
average_attempts = total_attempts / no_test_trials

print(f"Validation tests: {no_test_trials}")
print(f"Success rate: {success_rate:.2%}")
print(f"Average number of attempts: {average_attempts:.2f}")

In [ ]:
plt.figure(figsize=(12, 5))

# Plot 1: Success vs Failure Distribution
plt.subplot(1, 2, 1)
labels = ['Success', 'Failure']
counts = [success_history.count('Success'), success_history.count('Failure')]
colors = ['#2ecc71', '#e74c3c']

plt.pie(counts, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors, explode=(0.05, 0))
plt.title(f'Overall Success Rate ({success_rate:.2%})')

# Plot 2: Distribution of Attempts
plt.subplot(1, 2, 2)
# Setting discrete bins from 1 to max_attempts + 1
bins = np.arange(1, env.max_attempts + 2) - 0.5
plt.hist(attempts_per_episode, bins=bins, rwidth=0.8, color='skyblue', edgecolor='black', alpha=0.7)

plt.axvline(average_attempts, color='red', linestyle='--', linewidth=2, label=f'Avg: {average_attempts:.2f}')
plt.title('Distribution of Attempts per Episode')
plt.xlabel('Number of Attempts')
plt.ylabel('Episode Count')
plt.xticks(range(1, env.max_attempts + 1))
plt.grid(axis='y', alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig('wordle_evaluation_results.png')
plt.show()